# 20 — Opt-in Langfuse traces
The core workshop's local JSONL trace stays available without a server.
External traces can include prompts, so enable them only for approved data.



**Advanced follow-up to lesson 11 (observability).**


## Problem

A local event list cannot show all cross-service spans.


## Concept

Langfuse receives opt-in CrewAI traces through OpenTelemetry instrumentation.

OpenInference instruments CrewAI calls and sends spans to Langfuse only when enabled.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran   ◀ YOU ARE HERE
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
CrewAI run → local event trace; optional OpenInference spans → Langfuse
```

CrewAI operations → OpenInference spans → Langfuse. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Locate the local trace and inspect the remote opt-in flag

Locate the local trace and inspect the remote opt-in flag.


In [ ]:
import os
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.langfuse_tracing import enable_langfuse
from workshop.common import ensure_runtime

print("Local trace:", ensure_runtime() / "trace.jsonl")
print("Langfuse enabled:", os.getenv("WORKSHOP_LANGFUSE_ENABLED", "false"))



## Connect to Langfuse only with explicit permission and project keys
The integration instruments CrewAI calls made after it is enabled.



## Step 2 — Authenticate and instrument CrewAI for an approved Langfuse project

Authenticate and instrument CrewAI for an approved Langfuse project.


In [ ]:
if os.getenv("WORKSHOP_RUN_LIVE_DEMOS") == "true":
    print("Remote instrumentation active:", enable_langfuse())
else:
    print("Set WORKSHOP_RUN_LIVE_DEMOS=true to verify the configured Langfuse project.")


## Failure mode to know about

Remote traces can contain prompts and customer data. Export requires explicit data-owner approval.


## Production implication

Set access, retention, and redaction policy before remote export. Keep the local trace.


## Your turn

Confirm that a fictional run appears in your approved Langfuse project.

**Checkpoint:** No remote trace is exported when the opt-in flag is false.
